# Project Report: TripAdvisor Recommendation System

    Gabriel Matar and Octave Pedergnana

### Step 1: Data Loading and Preparation
This step involves loading the files and preparing the text. Before starting, I examine the CSV columns (and their metadata) to better understand the context. I also display the first few rows to better visualize the structure of the CSV.

In [ ]:
import pandas as pd
import numpy as np
import re
from rank_bm25 import BM25Okapi
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# So we load main datasets
df_reviews = pd.read_csv('reviews83325.csv')
df_places = pd.read_csv('Tripadvisor.csv')

print(df_reviews.info())
print(df_places.info())

print(df_reviews.head())
print(df_reviews.head())

C:\Users\gabri\AppData\Local\Temp\ipykernel_13144\3489190660.py:10: DtypeWarning: Columns (0: owner_langue, 1: owner_date_review, 2: owner_connection, 3: owner_responder, 4: owner_response, 5: owner_title) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv('reviews83325.csv')


<class 'pandas.DataFrame'>
RangeIndex: 340385 entries, 0 to 340384
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    340385 non-null  int64  
 1   idplace               340385 non-null  int64  
 2   titre                 340382 non-null  str    
 3   idauteur              339726 non-null  str    
 4   review                340385 non-null  str    
 5   note                  340385 non-null  int64  
 6   date_review           340385 non-null  str    
 7   date_visit            321147 non-null  str    
 8   langue                340385 non-null  str    
 9   published_platform    340385 non-null  str    
 10  typeReview            340385 non-null  str    
 11  subratings            340385 non-null  str    
 12  machine_translated    340385 non-null  bool   
 13  machine_translatable  340385 non-null  bool   
 14  owner_id              39641 non-null   float64
 15  owner_langu

We filter the reviews to keep only those in English and group all reviews for the same location together to create a single document per destination. We limit the number of reviews per location to prevent highly popular spots from skewing the results.

In [2]:
# Filter only "en" reviews (english)
df_reviews = df_reviews[df_reviews['langue'] == 'en'].copy()

# text cleaning function
def clean_text(text):
    if pd.isna(text): return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df_reviews['review_clean'] = df_reviews['review'].apply(clean_text)

# Aggregate reviews by place (joining on idplace)( We limit to 50 reviews per place) 
df_grouped = df_reviews.groupby('idplace')['review_clean'].apply(lambda x: " ".join(x[:50])).reset_index()

# And we merge
df_final = pd.merge(df_grouped, df_places, left_on='idplace', right_on='id')

### Step 2: Loading Metadata Dictionaries
Here, we use the other CSV files (cuisine.csv, AttractionSubCategorie.csv....). These files act as "dictionnaires" to convert the ID in the main file into more understandable names. Technically, it is not strictly necessary to load them because the models we will code check if the "keys" are identical, so there is no need to see the "values" in these CSV. However, it is still important to understand the data we are manipulating.

In [3]:
cuisine_df = pd.read_csv('cuisine.csv')
attraction_subcat_df = pd.read_csv('AttractionSubCategorie.csv')
attraction_subtype_df = pd.read_csv('AttractionSubType.csv')
restau_type_df = pd.read_csv('restaurantType.csv')

# Create dictionaries for an easy look
cuisine_map = dict(zip(cuisine_df['id'], cuisine_df['name']))
subcat_map = dict(zip(attraction_subcat_df['id'], attraction_subcat_df['name']))
subtype_map = dict(zip(attraction_subtype_df['id'], attraction_subtype_df['name']))
restau_type_map = dict(zip(restau_type_df['id'], restau_type_df['name']))

#We display
print(cuisine_map)
print(subcat_map)
print(subtype_map)
print(restau_type_map)

{4617: 'Italian', 5086: 'French', 5110: 'Mexican', 5379: 'Chinese', 5473: 'Japanese', 9908: 'American', 10345: 'Steakhouse', 10346: 'Indian', 10347: 'German', 10348: 'Brazilian', 10617: 'Belgian', 10618: 'Irish', 10620: 'Austrian', 10621: 'Brew Pub', 10622: 'Caribbean', 10626: 'Lebanese', 10627: 'Dutch', 10628: 'Swiss', 10631: 'Peruvian', 10632: 'African', 10633: 'Moroccan', 10634: 'Southwestern', 10635: 'Cajun & Creole', 10636: 'Philippine', 10637: 'Polish', 10639: 'Latin', 10640: 'Bar', 10641: 'Pizza', 10642: 'Cafe', 10643: 'Seafood', 10646: 'Fast food', 10648: 'International', 10649: 'Mediterranean', 10651: 'Barbecue', 10653: 'Sushi', 10654: 'European', 10655: 'Spanish', 10659: 'Asian', 10660: 'Thai', 10661: 'Korean', 10662: 'British', 10663: 'Turkish', 10664: 'Greek', 10665: 'Vegetarian Friendly', 10666: 'Deli', 10668: 'Grill', 10669: 'Contemporary', 10670: 'Pub', 10671: 'Fusion', 10675: 'Vietnamese', 10676: 'Diner', 10679: 'Healthy', 10680: 'Portuguese', 10681: 'Australian', 10682

### Step 3: Train / Test Split
Here, we split the locations into two halves (50/50). One half will serve as the reference database, and the other half will be used as queries to test whether our system correctly identifies the right neighbors.

In [4]:
# Randomly split between train (queries) and test (database) 
train_df, test_df = train_test_split(df_final, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

### Step 4: Reference Model (BM25 Baseline)
The BM25 model calculates the relevance between the words in a test review and the reviews in the reference group. It is a classic information retrieval algorithm. We will use it as a baseline to compare against the model we are going to create.

In [5]:
# Tokenize test for the model
tokenized_corpus = [doc.split() for doc in test_df['review_clean']]
bm25 = BM25Okapi(tokenized_corpus)

def get_bm25_recommendations(query_text):
    query_tokens = query_text.split()
    scores = bm25.get_scores(query_tokens)
    # Return indices of test_df sorted by relevance score
    return np.argsort(scores)[::-1]

### Step 5: Evaluation Functions (Ranking Error)
Level 1 simply checks if the type (Hotel, Restaurant...) is identical. Level 2 is stricter: it verifies if at least one sub-category (for example, Italian cuisine) matches between the query and the result. The error is calculated based on the position (n-1) of the first correct result found.

In [6]:
def parse_ids(value): #Helper to convert string ID to a set of integers
    if pd.isna(value) or value == "": return set()
    cleaned = re.sub(r'[\[\]\'\s]', '', str(value))
    if not cleaned: return set()
    return set(map(int, cleaned.split(',')))

def check_match_level2(q, r): #Check if at least one category matches for Level 2 evaluation
    if q['typeR'] != r['typeR']: return False
    
    # for Attractions
    if q['typeR'] in ['A', 'AP']:
        q_cats = parse_ids(q['activiteSubCategorie']) | parse_ids(q['activiteSubType'])
        r_cats = parse_ids(r['activiteSubCategorie']) | parse_ids(r['activiteSubType'])
        return not q_cats.isdisjoint(r_cats)
    
    #for Restaurants
    if q['typeR'] == 'R':
        q_rest = parse_ids(q['restaurantType']) | parse_ids(q['restaurantTypeCuisine'])
        r_rest = parse_ids(r['restaurantType']) | parse_ids(r['restaurantTypeCuisine'])
        return not q_rest.isdisjoint(r_rest)
    
    # for Hotels
    if q['typeR'] == 'H':
        return q['priceRange'] == r['priceRange']
    return False

def calculate_ranking_error(query_row, recommendations, test_set, level=1):
    for rank, idx in enumerate(recommendations):
        match = False
        if level == 1:
            match = (query_row['typeR'] == test_set.iloc[idx]['typeR'])
        else:
            match = check_match_level2(query_row, test_set.iloc[idx])
        
        if match: return rank # Error is n-1, so position 0 = 0 error 
    return None

### Step 6: Improved Model (TF-IDF + Cosine Similarity)
Now we are coding our new model. To outperform BM25, we are using a TF-IDF approach here. This allows us to give more weight to rare, characteristic words for a location (example: "mozzarella" for an Italian restaurant) while ignoring overly common words (like "cool," for example). We then compare the vectors using cosine similarity.

In [7]:
# Initialize TF IDF Vectorizer
tfidf_vec = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix_test = tfidf_vec.fit_transform(test_df['review_clean'])
tfidf_matrix_train = tfidf_vec.transform(train_df['review_clean'])

def get_tfidf_recommendations(query_idx):
    # Compute similarity between one query and all test documents
    query_vec = tfidf_matrix_train[query_idx]
    similarities = cosine_similarity(query_vec, tfidf_matrix_test).flatten()
    return np.argsort(similarities)[::-1]

### Step 7: Execution of the Full Evaluation
This is the final cell that calculates the 4 scores (2 for each level and each model). It uses the check_match_level2 function we defined earlier (the one that leverages category and price columns).

Note: Level 1 is a simple evaluation (for example, Hotel vs. Restaurant), but Level 2 is a more informative and non-binary evaluation. For the second level, a recommendation is considered successful if at least one category (like cuisine type or attraction sub-type) matches between the query location and the suggested location.

In [8]:
#result lists
results = {
    'BM25': {'L1': [], 'L2': []},
    'TF-IDF': {'L1': [], 'L2': []}
}

# Running evaluation on a sample (for example: 1000 queries)
n_queries = len(train_df) # finaly we decide to test with the max nb of queries 

for i in range(min(n_queries, len(train_df))):
    q_row = train_df.iloc[i]
    
    # Get recommendations from models
    recs_bm25 = get_bm25_recommendations(q_row['review_clean'])
    recs_tfidf = get_tfidf_recommendations(i)
    
    # evaluate BM25
    err_bm25_l1 = calculate_ranking_error(q_row, recs_bm25, test_df, level=1)
    err_bm25_l2 = calculate_ranking_error(q_row, recs_bm25, test_df, level=2)
    
    if err_bm25_l1 is not None: results['BM25']['L1'].append(err_bm25_l1)
    if err_bm25_l2 is not None: results['BM25']['L2'].append(err_bm25_l2)
    
    #evaluate TF-IDF
    err_tfidf_l1 = calculate_ranking_error(q_row, recs_tfidf, test_df, level=1)
    err_tfidf_l2 = calculate_ranking_error(q_row, recs_tfidf, test_df, level=2)
    
    if err_tfidf_l1 is not None: results['TF-IDF']['L1'].append(err_tfidf_l1)
    if err_tfidf_l2 is not None: results['TF-IDF']['L2'].append(err_tfidf_l2)

# Display final comparison table
for model in ['BM25', 'TF-IDF']:
    l1_avg = np.mean(results[model]['L1'])
    l2_avg = np.mean(results[model]['L2'])
    print(f"Model : {model} | Mean Error L1 : {l1_avg} | Mean Error L2 : {l2_avg:5f}")

Model : BM25 | Mean Error L1 : 0.6946564885496184 | Mean Error L2 : 6.396135
Model : TF-IDF | Mean Error L1 : 0.8778625954198473 | Mean Error L2 : 6.357488


### Results Analysis:

Level 1 Validation (Location Type): With an average error of 0.69 for BM25 and 0.87 for TF-IDF, both models almost always place a location of the correct type in the first or second position. BM25 remains slightly more performant here.

Level 2 Performance: This is where our goal was to design a model better than BM25. The TF-IDF model (6.35) achieves a lower error score than BM25 (6.39) by a very slim margin. This indicates that TF-IDF is more effective at recommending locations that share specific characteristics (Level 2).

Error System: For example, an error of 6 at Level 2 means that, on average, the user must scroll down to the 7th position (n+1) to find a location with a characteristic identical to their query.

Finally, we notice that our initial hypothesis is quite solid: similar experiences (hotels, restaurants, attractions) are indeed described in a similar manner by users.

;)